# Trading Bot Based On Harmonic Patterns And Fibonacci Levels

In [13]:
#Librerias
import pandas as pd
import numpy as np
import math

In [14]:
# Parámetros de trading
trade_size = 10000.00
ew_rate = 0.382
tp_rate = 0.618
sl_rate = -0.618


In [15]:
# === ZigZag básico (no repintante) ===
def zigzag(df):
    direction = 0
    zz = [np.nan] * len(df)
    for i in range(1, len(df)):
        is_up = df['close'].iloc[i] >= df['open'].iloc[i]
        is_down = df['close'].iloc[i] <= df['open'].iloc[i]
        if df['close'].iloc[i-1] >= df['open'].iloc[i-1] and is_down:
            direction = -1
            zz[i] = df['high'].iloc[i-1]
        elif df['close'].iloc[i-1] <= df['open'].iloc[i-1] and is_up:
            direction = 1
            zz[i] = df['low'].iloc[i-1]
    return pd.Series(zz, index=df.index)

In [16]:
# === Función para extraer los puntos X, A, B, C, D ===
def get_points(sz):
    points = sz.dropna()
    if len(points) < 5:
        return None
    return points.iloc[-5:].values  # X, A, B, C, D

In [17]:
# === Funciones auxiliares para ratios ===
def calculate_ratios(data):
    # data debe contener puntos X, A, B, C, D (o c, d) como precios
    X, A, B, C, D = data["X"], data["A"], data["B"], data["C"], data["D"]

    xab = abs((A - B) / (X - A)) if (X - A) != 0 else 0
    xad = abs((D - A) / (X - A)) if (X - A) != 0 else 0
    abc = abs((B - C) / (A - B)) if (A - B) != 0 else 0
    bcd = abs((C - D) / (B - C)) if (B - C) != 0 else 0

    return xab, xad, abc, bcd, C, D



In [18]:
# === Funciones de patrones ===
def isBat(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.382 <= xab <= 0.5 and
            0.382 <= abc <= 0.886 and
            1.618 <= bcd <= 2.618 and
            xad <= 1.0 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isAntiBat(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.500 <= xab <= 0.886 and
            1.000 <= abc <= 2.618 and
            1.618 <= bcd <= 2.618 and
            0.886 <= xad <= 1.000 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isAltBat(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (xab <= 0.382 and
            0.382 <= abc <= 0.886 and
            2.0 <= bcd <= 3.618 and
            xad <= 1.13 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isButterfly(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (xab <= 0.786 and
            0.382 <= abc <= 0.886 and
            1.618 <= bcd <= 2.618 and
            1.27 <= xad <= 1.618 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isAntiButterfly(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.236 <= xab <= 0.886 and
            1.130 <= abc <= 2.618 and
            1.000 <= bcd <= 1.382 and
            0.500 <= xad <= 0.886 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isABCD(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.382 <= abc <= 0.886 and
            1.13 <= bcd <= 2.618 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isGartley(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.5 <= xab <= 0.618 and
            0.382 <= abc <= 0.886 and
            1.13 <= bcd <= 2.618 and
            0.75 <= xad <= 0.875 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isAntiGartley(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.500 <= xab <= 0.886 and
            1.000 <= abc <= 2.618 and
            1.500 <= bcd <= 5.000 and
            1.000 <= xad <= 5.000 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isCrab(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.500 <= xab <= 0.875 and
            0.382 <= abc <= 0.886 and
            2.000 <= bcd <= 5.000 and
            1.382 <= xad <= 5.000 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isAntiCrab(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.250 <= xab <= 0.500 and
            1.130 <= abc <= 2.618 and
            1.618 <= bcd <= 2.618 and
            0.500 <= xad <= 0.750 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isShark(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.500 <= xab <= 0.875 and
            1.130 <= abc <= 1.618 and
            1.270 <= bcd <= 2.240 and
            0.886 <= xad <= 1.130 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isAntiShark(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.382 <= xab <= 0.875 and
            0.500 <= abc <= 1.000 and
            1.250 <= bcd <= 2.618 and
            0.500 <= xad <= 1.250 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def is5o(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (1.13 <= xab <= 1.618 and
            1.618 <= abc <= 2.24 and
            0.5 <= bcd <= 0.625 and
            0.0 <= xad <= 0.236 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isWolf(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (1.27 <= xab <= 1.618 and
            0 <= abc <= 5 and
            1.27 <= bcd <= 1.618 and
            0.0 <= xad <= 5 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isHnS(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (2.0 <= xab <= 10 and
            0.90 <= abc <= 1.1 and
            0.236 <= bcd <= 0.88 and
            0.90 <= xad <= 1.1 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isConTria(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (0.382 <= xab <= 0.618 and
            0.382 <= abc <= 0.618 and
            0.382 <= bcd <= 0.618 and
            0.236 <= xad <= 0.764 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))


def isExpTria(mode, data):
    xab, xad, abc, bcd, c, d = calculate_ratios(data)
    return (1.236 <= xab <= 1.618 and
            1.000 <= abc <= 1.618 and
            1.236 <= bcd <= 2.000 and
            2.000 <= xad <= 2.236 and
            ((mode == 1 and d < c) or (mode == -1 and d > c)))

In [19]:
# === Fib último punto ===
def f_last_fib(d, c, rate):
    fib_range = abs(d - c)
    return d - fib_range * rate if d > c else d + fib_range * rate

In [22]:
# Señales de patrones (suponiendo que ya definiste las funciones isBat, isAntiBat, etc.)
buy_patterns_00 = (
    isABCD(1, data) or isBat(1, data) or isAltBat(1, data) or isButterfly(1, data) or isGartley(1, data)
    or isCrab(1, data) or isShark(1, data) or is5o(1, data) or isWolf(1, data) or isHnS(1, data)
    or isConTria(1, data) or isExpTria(1, data)
)
buy_patterns_01 = (
    isAntiBat(1, data) or isAntiButterfly(1, data) or isAntiGartley(1, data)
    or isAntiCrab(1, data) or isAntiShark(1, data)
)
sel_patterns_00 = (
    isABCD(-1, data) or isBat(-1, data) or isAltBat(-1, data) or isButterfly(-1, data) or isGartley(-1, data)
    or isCrab(-1, data) or isShark(-1, data) or is5o(-1, data) or isWolf(-1, data) or isHnS(-1, data)
    or isConTria(-1, data) or isExpTria(-1, data)
)
sel_patterns_01 = (
    isAntiBat(-1, data) or isAntiButterfly(-1, data) or isAntiGartley(-1, data)
    or isAntiCrab(-1, data) or isAntiShark(-1, data)
)

NameError: name 'data' is not defined

In [ ]:

# Entradas y salidas
buy_entry = (buy_patterns_00 or buy_patterns_01) and close <= f_last_fib(ew_rate)
buy_close = high >= f_last_fib(tp_rate) or low <= f_last_fib(sl_rate)
sell_entry = (sel_patterns_00 or sel_patterns_01) and close >= f_last_fib(ew_rate)
sell_close = low <= f_last_fib(tp_rate) or high >= f_last_fib(sl_rate)


In [ ]:
# Variables persistentes
in_buy_trade = False
in_sell_trade = False



In [ ]:
# Ejecución de órdenes (ejemplo simple, aquí tendrías que integrarlo en tu framework de trading/backtesting)
if canTrade and buy_entry and not in_buy_trade:
    # Ejecutar compra
    print(f"BUY {trade_size} unidades")
    in_buy_trade = True

if canTrade and sell_entry and not in_sell_trade:
    # Ejecutar venta
    print(f"SELL {trade_size} unidades")
    in_sell_trade = True

# Cierre de posiciones
if in_buy_trade and buy_close:
    print("Cerrar compra")
    in_buy_trade = False

if in_sell_trade and sell_close:
    print("Cerrar venta")
    in_sell_trade = False



In [ ]:
# Alertas
buymsg = "Buy / Main, Close = close Symbol = symbol"
sellmsg = "Sell / Main, Close = close Symbol = symbol"
buyexit = "Buy exit"
sellexit = "Sell exit"

def get_close(message):
    return f"{message} Price: {close} Symbol: {symbol}"

if canTrade and buy_entry and not in_buy_trade:
    print("ALERTA:", get_close(buymsg))

if canTrade and sell_entry and not in_sell_trade:
    print("ALERTA:", get_close(sellmsg))

if in_buy_trade and buy_close:
    print("ALERTA:", get_close(buyexit))

if in_sell_trade and sell_close:
    print("ALERTA:", get_close(sellexit))


### Backtest

In [ ]:
# === Ejemplo de uso ===
df = pd.read_csv('data.csv')  # con columnas open, high, low, close, time
trades = run_strategy(df)
print(trades)